**SegFormer-B0 для набора France (разметка IGN CoSIA)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tester170/france-segformer-b0/blob/v1.0/France_SegFormer_B0.ipynb)

Блокнот разворачивает окружение эксперимента (Python 3.11, PyTorch 2.1.0 + CUDA 12.1, MMCV 2.1.0, MMEngine 0.10.7, MMSegmentation 1.2.2), скачивает обученную модель и набор France, проверяет модель штатным `tools/test.py` MMSegmentation, показывает результат на снимке и при желании дообучает модель.

Перед запуском выберите видеокарту: *Среда выполнения → Сменить среду выполнения → GPU*. Затем *Среда выполнения → Выполнить все*.

Модель распознаёт 14 классов покрытия земли по палитре CoSIA на ортофотоснимках с разрешением около 0,2 м. На валидационных фрагментах набора France она достигает mIoU 72,9 %, общей точности 88,6 % и κ = 0,860.

**1. Параметры**

Параметры ниже управляют разделами блокнота. Без изменений блокнот скачивает набор, проверяет модель и показывает пример, а обучение пропускает.

In [ ]:
import os
ROOT = os.environ.get('NB_ROOT', '/content')        # рабочая папка (в Colab — /content)
REPO = f'{ROOT}/france-segformer-b0'                 # код, конфигурации, веса
ENV = f'{ROOT}/py311'                                # окружение Python 3.11 эксперимента
RAW = f'{ROOT}/data/France'                          # исходный набор с Яндекс Диска (5,4 ГБ)
DATASET = True        # скачать и подготовить набор France (нужно для оценки и обучения)
EVALUATE = True       # проверить модель штатным tools/test.py на валидационных фрагментах
TRAIN = False         # дообучить модель (см. раздел 7)
TRAIN_ITERS = 2052    # число итераций дообучения: 513 итераций = 1 эпоха эксперимента
MY_IMAGES = f'{ROOT}/my_images'                      # свои снимки для распознавания (раздел 6), если папка есть
OUT = f'{ROOT}/results'
PY = f'{ENV}/bin/python'
os.makedirs(OUT, exist_ok=True)

**2. Видеокарта**

Проверка видеокарты. Без неё модель тоже работает, но во много раз медленнее.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "видеокарта не найдена: расчёт пойдёт на процессоре"

**3. Репозиторий и окружение**

Скачивается код проекта (версия v1.0) и создаётся отдельное окружение с точными версиями пакетов эксперимента. Окружение Colab при этом не меняется. Первый запуск занимает около 3–5 минут.

In [ ]:
if not os.path.isdir(REPO):
    !git clone --depth 1 --branch v1.0 https://github.com/tester170/france-segformer-b0.git {REPO}
!bash {REPO}/env/setup_env.sh {ENV}

**4. Модель и набор данных**

Веса обученной модели скачиваются из релиза v1.0 репозитория, контрольная сумма SHA-256 проверяется.

In [ ]:
!cd {REPO} && python3 scripts/fetch.py model

Набор France (16 фрагментов ортофотоснимков и разметка CoSIA, 5,4 ГБ) скачивается по публичной ссылке Яндекс Диска, каждый файл проверяется по SHA-256. Затем `scripts/prepare_data.py` раскладывает его в структуру MMSegmentation: 12 обучающих фрагментов режутся на плитки 2048 × 2048, 4 валидационных фрагмента сохраняются целиком. Скачивание занимает несколько минут, подготовка около 3 минут.

In [ ]:
if DATASET:
    !cd {REPO} && python3 scripts/fetch.py dataset --out {RAW}
    !cd {REPO} && {PY} scripts/prepare_data.py --src {RAW} --out data/france --workers 2

**5. Оценка модели**

Оценка выполняется штатным `tools/test.py` MMSegmentation 1.2.2 с конфигурацией репозитория: валидационные фрагменты целиком, скользящее окно 1024 × 1024 с шагом 768. Ожидаемый результат: mIoU 72,90 %, aAcc 88,62 %. На T4 оценка занимает около 2 минут.

In [ ]:
if DATASET and EVALUATE:
    !cd {REPO} && PYTHONPATH=. {PY} tools/test.py configs/france/segformer_mit-b0_1xb8-amp_france-1024x1024.py checkpoints/segformer_b0_france_ep76.pth --work-dir {OUT}/test 2>&1 | grep -v -E "Initialized|init_weights|Warning|warn"

**6. Пример распознавания**

Модель распознаёт один валидационный фрагмент (GeoTIFF из набора) скриптом `scripts/infer.py`. Результат сохраняется как маска классов CoSIA в GeoTIFF с исходной привязкой и как цветная маска.

In [ ]:
import glob
example = sorted(glob.glob(f'{RAW}/City/*48.95591_2.90781.tif'))
if example:
    !cd {REPO} && {PY} scripts/infer.py --input {example[0]} --out {OUT}/example --overlay

Снимок, разметка CoSIA и результат модели в уменьшенном виде. Цвета классов соответствуют палитре CoSIA.

In [ ]:
import json, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
Image.MAX_IMAGE_PIXELS = None
classes = json.load(open(f'{REPO}/splits/split_v1.json', encoding='utf-8'))['classes']
lut = np.zeros((256, 3), np.uint8)
for c in classes:
    lut[c['cosia_id']] = c['color']
if example:
    stem = os.path.basename(example[0])[:-4]
    small = lambda im: im.reduce(8)
    img = small(Image.open(example[0]).convert('RGB'))
    ref = Image.fromarray(lut[np.array(Image.open(example[0][:-4] + '_indexed.tif'))]).reduce(8)
    pred = small(Image.open(f'{OUT}/example/{stem}_color.png'))
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    for a, im, t in zip(ax, (img, ref, pred), ('Снимок', 'Разметка CoSIA', 'SegFormer-B0')):
        a.imshow(im); a.set_title(t); a.axis('off')
    fig.legend(handles=[Patch(color=np.array(c['color']) / 255, label=c['name_ru']) for c in classes],
               loc='lower center', ncol=7, frameon=False)
    plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

Свои снимки положите в папку `my_images` (в Colab — `/content/my_images`): GeoTIFF/TIFF, JP2, IMG, VRT, JPG или PNG, 8-битные RGB, лучше с разрешением около 0,2 м. Результаты появятся в `results/my_images`.

In [ ]:
if os.path.isdir(MY_IMAGES):
    !cd {REPO} && {PY} scripts/infer.py --input {MY_IMAGES} --out {OUT}/my_images --overlay
else:
    print('папки', MY_IMAGES, 'нет: шаг пропущен')

**7. Дообучение**

Обучение запускается штатным `tools/train.py` MMSegmentation с той же конфигурацией. Кодировщик инициализируется весами, предобученными на наборе LoveDA, как в эксперименте. Каждые 2 эпохи модель проверяется на валидационных фрагментах, лучшая сохраняется как `best_mIoU_iter_N.pth`.

Полное обучение (до 100 эпох по 513 итераций) на T4 займёт больше суток, поэтому в Colab по умолчанию выполняются 4 эпохи. Полное обучение лучше запускать на своём компьютере командой из README.

In [ ]:
if DATASET and TRAIN:
    !cd {REPO} && python3 scripts/fetch.py init
    !cd {REPO} && PYTHONPATH=. {PY} tools/train.py configs/france/segformer_mit-b0_1xb8-amp_france-1024x1024.py --work-dir {OUT}/train --cfg-options train_cfg.max_iters={TRAIN_ITERS} 2>&1 | grep -E "Iter|mIoU|best|Error"

**8. Результаты**

Архив с результатами (оценка, пример, свои снимки, обученная модель) скачивается на компьютер.

In [ ]:
import shutil
archive = shutil.make_archive(f'{ROOT}/france_results', 'zip', OUT)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass